# 🗺️ Módulo 2: El Orquestador de Misiones — DAG & Task Manager
## Notebook de Conocimiento Guiado

**Objetivo:** Dominar los Grafos Dirigidos Acíclicos (DAG), el ordenamiento topológico
y la planificación de tareas con prioridad, implementando un orquestador de misiones espaciales.

**Lore:** Eres el Comandante de la Flota Galáctica. Cada misión depende de otras.
Debes ejecutarlas en el orden correcto o la flota quedará varada.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | ¿Qué es un DAG? Aplicaciones reales |
| 🔨 Implementación | Detección de ciclos, Kahn, heapq |
| ✏️ Ejercicio 1 | `get_execution_waves()` — olas de paralelismo |
| ✏️ Ejercicio 2 | `critical_path_length()` — camino crítico |
| 🎯 Quiz | Preguntas de entrevista sobre DAGs |

## 📚 Parte 1: ¿Qué es un DAG?

Un **Grafo Dirigido Acíclico** (DAG) tiene tres propiedades:

```
DIRIGIDO: Las aristas tienen dirección  A → B  (A debe completarse antes que B)
ACÍCLICO: No hay ciclos               A → B → C → A  ← ¡PROHIBIDO!
```

**Aplicaciones reales:**
- **Apache Airflow / GitHub Actions / Make**: orquestación de pipelines
- **npm / pip / cargo**: resolución de dependencias de paquetes
- **Git**: el grafo de commits es un DAG
- **Compiladores**: orden de compilación de módulos

```
Ejemplo de DAG de misiones:
  Reparar_Motor
       ↓
  Repostar ← Comprar_Combustible
       ↓
  Despegar
```

La propiedad clave: **el orden topológico** garantiza que si existe la arista A→B,
entonces A aparece ANTES que B en el orden de ejecución.

In [ ]:
# 🔍 Visualización: ¿Por qué los ciclos son problemáticos?

# Un sistema con ciclo: A depende de B, B depende de A
# Esto es imposible de resolver: ¿cuál va primero?

grafo_con_ciclo = {
    "ComprarCombustible": set(),
    "RepararMotor": {"Despegar"},   # Motor requiere Despegar
    "Despegar": {"RepararMotor"},   # Despegar requiere Motor → ¡CICLO!
}

def tiene_ciclo_dfs(grafo):
    """Detecta ciclos con DFS y coloración de nodos (blanco/gris/negro)."""
    BLANCO, GRIS, NEGRO = 0, 1, 2
    color = {nodo: BLANCO for nodo in grafo}
    
    def dfs(nodo):
        color[nodo] = GRIS  # En progreso → si lo vemos de nuevo, hay ciclo
        for vecino in grafo.get(nodo, set()):
            if color[vecino] == GRIS:
                return True   # ¡Ciclo detectado!
            if color[vecino] == BLANCO and dfs(vecino):
                return True
        color[nodo] = NEGRO  # Completado
        return False
    
    return any(dfs(n) for n in grafo if color[n] == BLANCO)

print("¿Tiene ciclo?", tiene_ciclo_dfs(grafo_con_ciclo))  # True

grafo_valido = {
    "ComprarCombustible": set(),
    "RepararMotor": set(),
    "Repostar": {"ComprarCombustible", "RepararMotor"},
    "Despegar": {"Repostar"},
}
print("¿Tiene ciclo?", tiene_ciclo_dfs(grafo_valido))  # False

## 🏗️ Parte 2: Algoritmo de Kahn (Ordenamiento Topológico)

El algoritmo de Kahn usa BFS con in-degrees (grado de entrada):

```
1. Calcular in_degree[nodo] = número de dependencias de cada nodo
2. Cola inicial = todos los nodos con in_degree == 0 (sin dependencias)
3. Mientras la cola no esté vacía:
   a. Sacar nodo N
   b. Agregar N al orden de ejecución
   c. Para cada vecino V de N:
      - in_degree[V] -= 1
      - Si in_degree[V] == 0: agregar V a la cola
4. Si el orden tiene todos los nodos → éxito
   Si faltan nodos → el grafo tenía un ciclo
```

**Complejidad:** O(V + E) tiempo, O(V) espacio

In [ ]:
# 🔨 IMPLEMENTACIÓN: Algoritmo de Kahn
from collections import deque

def ordenamiento_topologico_kahn(dependencias):
    """
    Retorna la lista de nodos en orden topológico.
    dependencias: dict[str, set[str]]  →  {tarea: {prereqs}}
    """
    # Paso 1: Calcular in_degrees y construir grafo adjacente
    todos = set(dependencias.keys())
    for deps in dependencias.values():
        todos |= deps
    
    in_degree = {n: 0 for n in todos}
    adyacente = {n: [] for n in todos}   # n → [nodos que dependen de n]
    
    for tarea, prereqs in dependencias.items():
        for p in prereqs:
            adyacente[p].append(tarea)
            in_degree[tarea] += 1
    
    # Paso 2: Cola inicial — nodos sin dependencias
    cola = deque(n for n in todos if in_degree[n] == 0)
    orden = []
    
    while cola:
        nodo = cola.popleft()
        orden.append(nodo)
        for vecino in sorted(adyacente[nodo]):   # sorted para determinismo
            in_degree[vecino] -= 1
            if in_degree[vecino] == 0:
                cola.append(vecino)
    
    if len(orden) != len(todos):
        raise ValueError("El grafo tiene ciclos — orden topológico imposible")
    
    return orden

# Prueba con misiones galácticas
misiones = {
    "Despegar":          {"Repostar"},
    "Repostar":          {"ComprarCombustible", "RepararMotor"},
    "ComprarCombustible": set(),
    "RepararMotor":      set(),
}

orden = ordenamiento_topologico_kahn(misiones)
print("Orden de ejecución:")
for i, m in enumerate(orden, 1):
    print(f"  {i}. {m}")

## 🏗️ Parte 3: Cola de Prioridad con heapq

Cuando varias tareas están listas al mismo tiempo, elegimos la de **mayor prioridad**.
Python's `heapq` es un *min-heap*: el elemento con **menor valor** sale primero.

In [ ]:
# 🔨 IMPLEMENTACIÓN COMPLETA: TaskManager simplificado
import heapq
from enum import Enum
from dataclasses import dataclass, field

class Estado(Enum):
    PENDIENTE = "pendiente"
    LISTA = "lista"
    COMPLETADA = "completada"
    CANCELADA = "cancelada"

@dataclass
class Tarea:
    id: str
    nombre: str
    prioridad: int = 0
    estado: Estado = Estado.PENDIENTE
    dependencias: set = field(default_factory=set)
    
    def __lt__(self, other):
        return (self.prioridad, self.id) < (other.prioridad, other.id)

class OrquestadorSimple:
    def __init__(self):
        self.tareas = {}
        self.dependientes = {}   # dependientes[A] = {tareas que dependen de A}
    
    def agregar(self, tid, nombre, prioridad=0, deps=None):
        t = Tarea(tid, nombre, prioridad, dependencies=set(deps or []))
        self.tareas[tid] = t
        self.dependientes.setdefault(tid, set())
        for d in (deps or []):
            self.dependientes.setdefault(d, set()).add(tid)
        self._actualizar_lista(tid)
        return t
    
    def _actualizar_lista(self, tid):
        t = self.tareas[tid]
        if t.estado == Estado.PENDIENTE:
            if all(self.tareas[d].estado == Estado.COMPLETADA
                   for d in t.dependencias if d in self.tareas):
                t.estado = Estado.LISTA
    
    def siguiente(self):
        cola = [t for t in self.tareas.values() if t.estado == Estado.LISTA]
        return heapq.nsmallest(1, cola)[0] if cola else None
    
    def completar(self, tid):
        self.tareas[tid].estado = Estado.COMPLETADA
        for dep in self.dependientes.get(tid, set()):
            self._actualizar_lista(dep)
    
    def cancelar(self, tid):
        """Cancelación en cascada."""
        if self.tareas[tid].estado == Estado.CANCELADA: return
        self.tareas[tid].estado = Estado.CANCELADA
        for dep in self.dependientes.get(tid, set()):
            self.cancelar(dep)

# Demo
orq = OrquestadorSimple()
orq.agregar("combustible", "Comprar Combustible", prioridad=1)
orq.agregar("motor", "Reparar Motor", prioridad=2)
orq.agregar("repostar", "Repostar", prioridad=1, deps=["combustible", "motor"])
orq.agregar("despegar", "Despegar", prioridad=0, deps=["repostar"])

print("Siguiente tarea:", orq.siguiente().nombre)
orq.completar("combustible")
orq.completar("motor")
print("Siguiente tarea:", orq.siguiente().nombre)

## ✏️ Ejercicio 1: `get_execution_waves()`

**Problema:** Implementa una función que agrupe las tareas en "olas" de ejecución paralela.
Una ola contiene todas las tareas que pueden ejecutarse simultáneamente (todas sus dependencias
están en olas anteriores).

```python
# Ejemplo:
# Ola 1: [ComprarCombustible, RepararMotor]   ← sin dependencias
# Ola 2: [Repostar]                           ← depende de ola 1
# Ola 3: [Despegar]                           ← depende de ola 2
```

**Tu tarea:** Completa la función `get_execution_waves()`

In [ ]:
# ✏️ EJERCICIO 1 — Completa el TODO

def get_execution_waves(dependencias):
    """
    Retorna lista de listas: cada sublista es una 'ola' de tareas paralelas.
    
    Args:
        dependencias: dict[str, set[str]] — {tarea: {prereqs}}
    
    Returns:
        list[list[str]] — olas de ejecución
    """
    # TODO: Implementa usando el algoritmo de Kahn modificado
    # Pista: cuando varios nodos llegan a in_degree==0 al mismo tiempo,
    # todos pertenecen a la misma ola
    pass

# Test básico — debería retornar 3 olas
resultado = get_execution_waves({
    "ComprarCombustible": set(),
    "RepararMotor": set(),
    "Repostar": {"ComprarCombustible", "RepararMotor"},
    "Despegar": {"Repostar"},
})
print("Olas:", resultado)
# Esperado: [['ComprarCombustible', 'RepararMotor'], ['Repostar'], ['Despegar']]

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def get_execution_waves(dependencias):
    todos = set(dependencias.keys())
    for deps in dependencias.values():
        todos |= deps
    
    in_degree = {n: 0 for n in todos}
    adyacente = {n: [] for n in todos}
    
    for tarea, prereqs in dependencias.items():
        for p in prereqs:
            adyacente[p].append(tarea)
            in_degree[tarea] += 1
    
    olas = []
    cola_actual = sorted(n for n in todos if in_degree[n] == 0)
    
    while cola_actual:
        olas.append(sorted(cola_actual))
        siguiente_ola = []
        for nodo in cola_actual:
            for vecino in adyacente[nodo]:
                in_degree[vecino] -= 1
                if in_degree[vecino] == 0:
                    siguiente_ola.append(vecino)
        cola_actual = siguiente_ola
    
    return olas

resultado = get_execution_waves({
    "ComprarCombustible": set(),
    "RepararMotor": set(),
    "Repostar": {"ComprarCombustible", "RepararMotor"},
    "Despegar": {"Repostar"},
})
print("Olas de ejecución:")
for i, ola in enumerate(resultado, 1):
    print(f"  Ola {i}: {ola}")

## ✏️ Ejercicio 2: `critical_path_length()`

El **camino crítico** en un DAG es el camino más largo (en número de nodos o en peso).
Determina la duración mínima del proyecto completo, incluso con paralelismo.

**Tu tarea:** Dado un grafo de dependencias y duraciones, encuentra la longitud del camino crítico.

In [ ]:
# ✏️ EJERCICIO 2 — Completa el TODO

def critical_path_length(dependencias, duraciones=None):
    """
    Calcula la duración del camino crítico.
    
    Args:
        dependencias: dict[str, set[str]]
        duraciones: dict[str, int] — duración de cada tarea (default: 1 para todos)
    
    Returns:
        int — duración del camino crítico
    """
    if duraciones is None:
        todos = set(dependencias.keys())
        for d in dependencias.values():
            todos |= d
        duraciones = {t: 1 for t in todos}
    
    # TODO: Usa programación dinámica sobre el DAG
    # earliest_finish[t] = max(earliest_finish[dep] for dep in deps) + duracion[t]
    pass

# Test
deps = {
    "A": set(), "B": set(),
    "C": {"A"}, "D": {"B"},
    "E": {"C", "D"},
}
print("Camino crítico:", critical_path_length(deps, {"A":1,"B":3,"C":2,"D":1,"E":1}))
# A→C→E = 1+2+1 = 4
# B→D→E = 3+1+1 = 5  ← camino crítico
# Esperado: 5

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

def critical_path_length(dependencias, duraciones=None):
    todos = set(dependencias.keys())
    for d in dependencias.values():
        todos |= d
    if duraciones is None:
        duraciones = {t: 1 for t in todos}
    
    # Ordenamiento topológico primero
    in_degree = {n: 0 for n in todos}
    adyacente = {n: [] for n in todos}
    for tarea, prereqs in dependencias.items():
        for p in prereqs:
            adyacente[p].append(tarea)
            in_degree[tarea] += 1
    
    from collections import deque
    cola = deque(n for n in todos if in_degree[n] == 0)
    orden = []
    while cola:
        n = cola.popleft()
        orden.append(n)
        for v in adyacente[n]:
            in_degree[v] -= 1
            if in_degree[v] == 0:
                cola.append(v)
    
    # DP sobre el orden topológico
    earliest = {n: duraciones.get(n, 1) for n in todos}
    for n in orden:
        for v in adyacente[n]:
            earliest[v] = max(earliest[v], earliest[n] + duraciones.get(v, 1))
    
    return max(earliest.values())

print("Camino crítico:", critical_path_length(
    {"A": set(), "B": set(), "C": {"A"}, "D": {"B"}, "E": {"C", "D"}},
    {"A":1, "B":3, "C":2, "D":1, "E":1}
))  # 5

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Cuándo usarías BFS (Kahn) vs DFS para el ordenamiento topológico?
> **R:** Kahn (BFS) es mejor cuando necesitas detectar el nivel de cada nodo (olas de paralelismo).
> DFS es más simple de implementar y requiere menos memoria auxiliar.

**P2:** En Airflow, ¿qué error se produce cuando defines dependencias circulares?
> **R:** `AirflowDagCycleException`. Airflow valida el DAG al cargarlo.

**P3:** ¿Cómo representarías un DAG en memoria? ¿Lista de adyacencia vs matriz de adyacencia?
> **R:** Lista de adyacencia: O(V+E) espacio, mejor para grafos dispersos (la mayoría).
> Matriz: O(V²) espacio, mejor para grafos densos o cuando necesitas O(1) para verificar aristas.

**P4:** ¿Cuál es la complejidad del ordenamiento topológico de Kahn?
> **R:** O(V + E) tiempo, O(V) espacio adicional. Lineal en el tamaño del grafo.

**P5:** ¿Cómo detectarías si dos tareas pueden ejecutarse en paralelo?
> **R:** Si están en la misma "ola" del algoritmo de Kahn — ninguna depende de la otra
> ni siquiera transitivamente. También se puede resolver con LCA (Lowest Common Ancestor).